# Lab 3 — Compliance: PII Scrubbing & Audit Logging

**Goal:** Make your agent GDPR/HIPAA-ready: scrub PII before it reaches the LLM, and keep an immutable audit trail.

**Legal context:**  
Under GDPR Article 25 ('privacy by design'), you must not send more personal data than necessary to a third-party processor (OpenAI/Anthropic counts as one). The audit log is required for demonstrating compliance in a regulatory review.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from dotenv import load_dotenv
load_dotenv(override=True)
from middleware import PIIScrubber, AuditLogger
from openai import OpenAI

client = OpenAI()

def base_agent(prompt: str, **kwargs) -> str:
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=150,
    )
    return resp.choices[0].message.content

print('Ready ✓')

## 1. PII detection and scrubbing

In [ ]:
scrubber = PIIScrubber(base_agent)

test_inputs = [
    'My email is jane.smith@company.com, please reset my account.',
    'My SSN is 123-45-6789 and my card is 4111111111111111.',
    'Call me at (555) 867-5309 or (800) 555-0199.',
    'I live at 123 Main Street, Austin TX.',
    'How do I configure a Python virtual environment?',  # no PII
]

for text in test_inputs:
    cleaned, count = scrubber.scrub(text)
    print(f'Input:   {text}')
    print(f'Cleaned: {cleaned}')
    print(f'Redacted: {count} item(s)')
    print()

## 2. Confirm PII doesn't reach the LLM

In [ ]:
# Track what the LLM actually receives
received_prompts = []

def tracking_agent(prompt: str, **kwargs) -> str:
    received_prompts.append(prompt)
    return base_agent(prompt, **kwargs)

tracking_scrubber = PIIScrubber(tracking_agent)
result = tracking_scrubber('Please help john.doe@example.com reset their password. Their phone is 555-123-4567.')

print('What the LLM received:')
print(received_prompts[-1])

print('\nVerification — no PII leaked:')
pii_leaked = any(x in received_prompts[-1] for x in ['john.doe', 'example.com', '555-123'])
print('PII in LLM prompt:', pii_leaked, '(should be False)')

## 3. Audit logger

In [ ]:
import tempfile, os
log_path = os.path.join(tempfile.gettempdir(), 'agent_audit.jsonl')

audited_agent = AuditLogger(PIIScrubber(base_agent), log_file=log_path)

queries = [
    ('alice', 'How do I export my data?'),
    ('bob', 'My email is bob@test.com, help me update it.'),
    ('alice', 'What are your pricing plans?'),
]

for user_id, query in queries:
    result = audited_agent(query, user_id=user_id)
    print(f'[{user_id}] {result[:60]}')

print(f'\nAudit log written to: {log_path}')

In [ ]:
import json

print('Audit log contents:')
with open(log_path) as f:
    for line in f:
        entry = json.loads(line)
        print(f"  {entry['entry_id'][:8]}  user={entry['user_id']:6}  hash={entry['prompt_hash']}  lat={entry['latency_seconds']:.2f}s")

print(f'\nCompliance summary:')
print(json.dumps(audited_agent.compliance_summary(), indent=2))

## 4. GDPR right-to-forget simulation
Under GDPR, users can request deletion of their data. Here we show how to scrub a user's log entries.

In [ ]:
def gdpr_forget(log_path: str, user_id: str) -> int:
    """Remove all audit entries for a specific user. Returns count deleted."""
    import tempfile, shutil
    
    tmp_path = log_path + '.tmp'
    deleted = 0
    
    with open(log_path) as infile, open(tmp_path, 'w') as outfile:
        for line in infile:
            entry = json.loads(line)
            if entry.get('user_id') == user_id:
                deleted += 1
            else:
                outfile.write(line)
    
    shutil.move(tmp_path, log_path)
    return deleted


before_count = sum(1 for _ in open(log_path))
deleted = gdpr_forget(log_path, 'bob')
after_count = sum(1 for _ in open(log_path))

print(f'GDPR forget for user "bob":')
print(f'  Before: {before_count} entries')
print(f'  Deleted: {deleted} entries')
print(f'  After: {after_count} entries')

## Exercise
Add a custom PII pattern to `PIIScrubber` for **UK National Insurance numbers** (format: two letters, six digits, one letter — e.g. AB123456C).  
Test that it correctly redacts NI numbers while leaving unrelated alphanumeric strings intact.

In [ ]:
# Your code here
